# RAM-H1200 Bone Segmentation - Kaggle Experiment
### DenseNet121 -> LayerCAM -> Bone Morphology -> SAM ViT-B -> Evaluation / U-Net

This notebook is an executable experiment report. It is designed to be run from top to bottom while also explaining the full pipeline.

Every major stage includes:

- the purpose of the stage;
- its inputs and outputs;
- the exact command being executed;
- visual checks for image-processing outputs.

Dataset: **RAM-H1200 only**. The segmentation target is the visible bone region in hand X-ray images, not the full hand or soft-tissue silhouette.

## 0. Pipeline Overview

```text
RAM-H1200 hand X-ray
  -> Stage 1: DenseNet121 hand checkpoint
  -> Stage 2: LayerCAM localization
  -> Stage 3: bone-specific morphology
  -> Stage 4: SAM box/point prompting
  -> Stage 5: bone-aware mask selection and conservative refinement
  -> pseudo bone mask
  -> Dice/IoU against RAM-H1200 ground truth

Optional baseline:
RAM-H1200 image + GT mask -> U-Net -> supervised segmentation result
```

RAM-H1200 provides ground-truth bone masks, so the pseudo-mask branch can be evaluated quantitatively, not only visually.

## 1. Kaggle Paths and Run Switches

This cell defines repository paths, output folders, image size, and which expensive stages should run.

Recommended first run:

- keep `RUN_TRAIN_CLASSIFIER=True` if no compatible `best_classifier.pt` is attached;
- keep `RUN_FULL_PSEUDO_MASKS=False` for a quick 10-image preview;
- after the preview looks reasonable, set `RUN_FULL_PSEUDO_MASKS=True` and rerun Stage 2 + Stage 7.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

KAGGLE_INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')

REPO_URL = 'https://github.com/itsthang333/Thesis.git'
BRANCH = 'TN_exp'
PROJECT_PARENT = WORKING / 'Thesis'
PROJECT_DIR = PROJECT_PARENT / 'project'

OUTPUT_ROOT = WORKING / 'ThesisOutputs'
SAM_CHECKPOINT = OUTPUT_ROOT / 'checkpoints' / 'sam_vit_b_01ec64.pth'

CLASSIFIER_OUTPUT = OUTPUT_ROOT / 'classifier'
PSEUDO_OUTPUT = OUTPUT_ROOT / 'pseudo_masks'
SEG_OUTPUT = OUTPUT_ROOT / 'segmentation'
EVAL_CSV = OUTPUT_ROOT / 'ramh1200_eval.csv'
VIZ_OUTPUT = OUTPUT_ROOT / 'viz'
DEBUG_OUTPUT = OUTPUT_ROOT / 'debug_viz'
MORPH_DEBUG_OUTPUT = OUTPUT_ROOT / 'morphology_debug'
ABLATION_OUTPUT = OUTPUT_ROOT / 'ablation_preview'

IMAGE_SIZE = 384
NUM_WORKERS = 2
BATCH_SIZE_CLASSIFIER = 4
BATCH_SIZE_SEGMENTATION = 4
EPOCHS_CLASSIFIER = 25
EPOCHS_SEGMENTATION = 25

RUN_TRAIN_CLASSIFIER = True
RUN_FULL_PSEUDO_MASKS = False
RUN_EVALUATE_PSEUDO = True
RUN_TRAIN_UNET = False
RUN_VISUALIZE_SAMPLE = True
RUN_MORPHOLOGY_EXPLAINER = True
RUN_DEBUG_SAM = True
RUN_ABLATION_PREVIEW = False

DATASET_OVERRIDE = ''
HF_REPO_ID = 'TokyoTechMagicYang/RAM-H1200-v1'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

## 2. Kaggle Environment Setup

This stage prepares the runtime:

1. clone the `TN_exp` branch;
2. install only the extra packages needed by the project;
3. handle the Kaggle P100/PyTorch compatibility issue before importing `torch`;
4. verify that the expected project scripts exist.

In [ ]:
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_PARENT)], check=True)
else:
    print('Project already exists:', PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())

TORCH_ALREADY_IMPORTED = 'torch' in sys.modules

def run_pip(args, description):
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f'Failed to install {description}')
    print(f'Installed: {description}')

try:
    gpu_query = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
    GPU_NAME = gpu_query.stdout.strip()
except Exception:
    GPU_NAME = ''
print(f'GPU: {GPU_NAME or "not detected"}')

if 'P100' in GPU_NAME:
    if TORCH_ALREADY_IMPORTED:
        raise RuntimeError('Torch was already imported. Restart the Kaggle session and run again from the first cell.')
    print('Installing PyTorch 2.5.1 CUDA 11.8 with Pascal/sm_60 support...')
    run_pip(['--no-cache-dir', '--force-reinstall', 'torch==2.5.1', '--index-url', 'https://download.pytorch.org/whl/cu118'], 'PyTorch 2.5.1 CUDA 11.8')
    run_pip(['--no-cache-dir', '--force-reinstall', '--no-deps', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu118'], 'torchvision 0.20.1')

run_pip(['pycocotools'], 'pycocotools')
run_pip(['opencv-python'], 'opencv-python')
run_pip(['--no-deps', 'git+https://github.com/facebookresearch/segment-anything.git'], 'segment-anything')

import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))
    print('CUDA runtime:', torch.version.cuda)

required_files = [
    PROJECT_DIR / 'datasets' / 'ramh1200.py',
    PROJECT_DIR / 'train_classifier.py',
    PROJECT_DIR / 'generate_pseudo_masks.py',
    PROJECT_DIR / 'evaluate_ramh1200_masks.py',
    PROJECT_DIR / 'train_segmentation.py',
    PROJECT_DIR / 'visualize_pipeline.py',
]
for path in required_files:
    print(('OK  ' if path.exists() else 'MISS'), path)

## 3. Resolve RAM-H1200 Dataset

The notebook first searches attached Kaggle Inputs. If RAM-H1200 is not attached and Internet is enabled, it downloads the dataset from Hugging Face.

Expected layout:

```text
RAM-H1200-v1/
  Segmentation/
    train/
    val/ or validation/
    test/
```

In [ ]:
def has_ram_layout(path: Path) -> bool:
    return (path / 'Segmentation').exists() or ((path / 'train').exists() and (path / 'train' / '_annotations_bone_rle.coco.json').exists())

def find_ram_root(base: Path):
    candidates = []
    if base and base.exists():
        candidates.append(base)
        candidates.extend(base.glob('*'))
        candidates.extend(base.glob('*/*'))
    for candidate in candidates:
        if (candidate / 'Segmentation').exists():
            return candidate
        if (candidate / 'train' / '_annotations_bone_rle.coco.json').exists():
            return candidate
    return None

print('Attached Kaggle inputs:')
if KAGGLE_INPUT.exists():
    for entry in sorted(KAGGLE_INPUT.iterdir()):
        print(' -', entry)

RAM_ROOT = find_ram_root(Path(DATASET_OVERRIDE)) if DATASET_OVERRIDE else find_ram_root(KAGGLE_INPUT)

if RAM_ROOT is None:
    print('RAM-H1200 was not found under /kaggle/input. Trying Hugging Face download...')
    run_pip(['huggingface_hub'], 'huggingface_hub')
    from huggingface_hub import snapshot_download
    RAM_ROOT = Path(snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type='dataset',
        local_dir=str(WORKING / 'RAM-H1200-v1'),
        local_dir_use_symlinks=False,
    ))

print('RAM_ROOT:', RAM_ROOT)
if not has_ram_layout(RAM_ROOT):
    raise FileNotFoundError(f'RAM-H1200 segmentation layout was not found at {RAM_ROOT}')

## 4. Dataset Inspection and Ground-Truth Visualization

**Purpose:** verify that the notebook reads the correct dataset and that the annotation target is bone segmentation.

**Input:** RAM-H1200 split folders and COCO RLE annotations.

**Visual output:** original X-ray, GT bone mask, and GT overlay.

In [ ]:
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

SEG_ROOT = RAM_ROOT / 'Segmentation' if (RAM_ROOT / 'Segmentation').exists() else RAM_ROOT
if not (SEG_ROOT / 'train').exists():
    raise FileNotFoundError(f'Train split was not found under {SEG_ROOT}')

print('SEG_ROOT:', SEG_ROOT)
for split in ['train', 'val', 'validation', 'test']:
    split_dir = SEG_ROOT / split
    if split_dir.exists():
        ann_path = split_dir / '_annotations_bone_rle.coco.json'
        images = sorted([p for p in split_dir.iterdir() if p.suffix.lower() in {'.bmp', '.png', '.jpg', '.jpeg', '.tif', '.tiff'}])
        print(f'{split:<10} images={len(images):>5} annotation={ann_path.exists()}')
        if ann_path.exists():
            data = json.loads(ann_path.read_text(encoding='utf-8'))
            cats = [c.get('name', c.get('id')) for c in data.get('categories', [])]
            print('  categories:', cats[:30])
            print('  coco images:', len(data.get('images', [])), 'annotations:', len(data.get('annotations', [])))

from datasets.ramh1200 import RAMH1200SegmentationDataset
preview_ds = RAMH1200SegmentationDataset(root=RAM_ROOT, split='val', image_size=IMAGE_SIZE)
image_tensor, mask_tensor, image_name = preview_ds[0]
sample = preview_ds.samples[0]
image_path = Path(sample['image_path'])

original = Image.open(image_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
gt_mask = mask_tensor[0].numpy() > 0.5
original_np = np.array(original)
overlay = original_np.copy()
overlay[gt_mask] = (0.55 * overlay[gt_mask] + 0.45 * np.array([255, 40, 40])).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(original_np)
axes[0].set_title(f'Original image\\n{image_name}')
axes[1].imshow(gt_mask, cmap='gray')
axes[1].set_title('Ground-truth bone mask')
axes[2].imshow(overlay)
axes[2].set_title('GT mask overlay')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print('image tensor:', tuple(image_tensor.shape), image_tensor.dtype)
print('mask tensor:', tuple(mask_tensor.shape), mask_tensor.dtype, 'mask pixels:', int(mask_tensor.sum().item()))

## 5. SAM Checkpoint

**Purpose:** SAM ViT-B is used in Stage 2 to refine component boxes and points into candidate masks.

If the checkpoint is not attached, this cell downloads it.

In [ ]:
if not SAM_CHECKPOINT.exists():
    print('SAM checkpoint was not found. Downloading SAM ViT-B...')
    import urllib.request
    SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    url = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
    urllib.request.urlretrieve(url, SAM_CHECKPOINT)
else:
    print('SAM checkpoint exists:', SAM_CHECKPOINT)

## 6. Stage 1: DenseNet121 Hand Checkpoint

**Pipeline role:** provide features and gradients for LayerCAM.

**Input:** RAM-H1200 hand X-ray images.

**Training target:** `hand = 1`.

**Output:** `best_classifier.pt` and classifier training curves.

In [ ]:
classifier_cmd = [
    sys.executable, 'train_classifier.py',
    '--ram-root', str(RAM_ROOT),
    '--train-split', 'train',
    '--val-split', 'val',
    '--target-columns', 'hand',
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE_CLASSIFIER),
    '--num-workers', str(NUM_WORKERS),
    '--epochs', str(EPOCHS_CLASSIFIER),
    '--output-dir', str(CLASSIFIER_OUTPUT),
]
print(' '.join(classifier_cmd))
if RUN_TRAIN_CLASSIFIER:
    subprocess.run(classifier_cmd, check=True)
else:
    print('RUN_TRAIN_CLASSIFIER=False; classifier training is skipped.')

In [ ]:
import pandas as pd

CLASSIFIER_CHECKPOINT = CLASSIFIER_OUTPUT / 'best_classifier.pt'
log_path = CLASSIFIER_OUTPUT / 'training_log.csv'
print('Classifier checkpoint:', CLASSIFIER_CHECKPOINT, 'exists=', CLASSIFIER_CHECKPOINT.exists())
if log_path.exists():
    df = pd.read_csv(log_path)
    display(df.tail())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df[['train_loss', 'val_loss']].plot(ax=axes[0], title='Classifier loss')
    df[['train_f1', 'val_f1']].plot(ax=axes[1], title='Classifier F1')
    for ax in axes:
        ax.set_xlabel('epoch')
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Classifier training_log.csv was not found yet.')

## 7. Stage 2: Pseudo-Mask Generation with LayerCAM + Bone Morphology + SAM

**Pipeline role:** generate a bone pseudo mask from an X-ray image.

**Internal stages:**

1. DenseNet121 produces gradients and features.
2. LayerCAM localizes relevant hand/bone evidence.
3. Bone morphology combines radiographic intensity, cortical edges, and CAM.
4. Connected components generate boxes and structured positive points.
5. SAM proposes candidate masks.
6. `bone_hybrid` scoring selects the best masks.
7. Conservative morphology cleans the final pseudo mask.

In [ ]:
if not CLASSIFIER_CHECKPOINT.exists():
    raise FileNotFoundError(f'Missing classifier checkpoint: {CLASSIFIER_CHECKPOINT}. Run Stage 1 first.')

process_args = ['--process-all'] if RUN_FULL_PSEUDO_MASKS else ['--max-images', '10']
pseudo_cmd = [
    sys.executable, 'generate_pseudo_masks.py',
    '--ram-root', str(RAM_ROOT),
    '--split', 'val',
    '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
    '--sam-checkpoint', str(SAM_CHECKPOINT),
    '--target-columns', 'hand',
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', '1',
    '--num-workers', str(NUM_WORKERS),
    '--confidence-threshold', '0.5',
    '--cam-percentile', '85.0',
    '--max-points', '5',
    '--min-component-area', '100',
    '--mask-score-threshold', '0.4',
    '--selection-method', 'bone_hybrid',
    '--fusion-topk', '3',
    '--morphology-fusion-mode', 'components',
    '--sam-prompt-mode', 'box_point',
    '--max-bone-components', '6',
    '--points-per-component', '3',
    '--bbox-padding-ratio', '0.05',
    '--negative-points-per-component', '0',
    '--bone-seed-percentile', '88',
    '--bone-support-percentile', '62',
    '--closing-kernel', '5',
    '--opening-kernel', '0',
    '--max-hole-area', '500',
    '--min-size', '40',
    '--save-visuals-limit', '10',
    *process_args,
    '--output-dir', str(PSEUDO_OUTPUT),
]
print(' '.join(pseudo_cmd))
subprocess.run(pseudo_cmd, check=True)

### 7.1 Visual Check: Original Image, LayerCAM Overlay, and Pseudo Mask

Inspect this before running a full split. The CAM should activate around bone structures, and the pseudo mask should not become a full soft-tissue hand silhouette.

In [ ]:
mask_dir = PSEUDO_OUTPUT / 'masks'
overlay_dir = PSEUDO_OUTPUT / 'overlays'
mask_files = sorted(mask_dir.glob('*.png'))
print('Number of pseudo masks:', len(mask_files), mask_dir)

n = min(5, len(mask_files))
if n:
    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
    if n == 1:
        axes = np.array([axes])
    val_dir = SEG_ROOT / 'val' if (SEG_ROOT / 'val').exists() else SEG_ROOT / 'validation'
    for row, mask_path in zip(axes, mask_files[:n]):
        stem = mask_path.stem
        image_match = next(iter(sorted(val_dir.glob(f'{stem}.*'))), None)
        overlay_match = next(iter(sorted(overlay_dir.glob(f'{stem}*fused_layercam.png'))), None)
        if image_match:
            row[0].imshow(Image.open(image_match).convert('RGB'))
            row[0].set_title(f'Original\\n{stem}')
        if overlay_match:
            row[1].imshow(Image.open(overlay_match).convert('RGB'))
            row[1].set_title('LayerCAM overlay')
        row[2].imshow(Image.open(mask_path), cmap='gray')
        row[2].set_title('Pseudo mask')
        for ax in row:
            ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No pseudo masks were found.')

### 7.2 Image-Processing Explainer Before SAM

This cell visualizes the image-processing path before SAM:

- original X-ray;
- LayerCAM;
- CAM + bone guidance prompt map;
- enhanced grayscale;
- cortical edge response;
- bone likelihood;
- seed pixels;
- morphology support;
- selected prompt points.

In [ ]:
if RUN_MORPHOLOGY_EXPLAINER and CLASSIFIER_CHECKPOINT.exists():
    import torch
    from datasets.common import make_classification_transform
    from models.classifier import DenseNet121AnatomyClassifier
    from models.layercam import LayerCAM
    from pseudo.generate_layercam import generate_fused_cam
    from pseudo.bone_morphology import build_bone_guidance, fuse_cam_with_bone_guidance
    from pseudo.extract_prompts import extract_point_prompts
    from pseudo.visualization import tensor_to_pil

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(CLASSIFIER_CHECKPOINT, map_location='cpu')
    target_columns = checkpoint.get('target_columns', ['hand'])
    model = DenseNet121AnatomyClassifier(num_classes=len(target_columns), pretrained=False)
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    model.to(device).eval()

    val_dir = SEG_ROOT / 'val' if (SEG_ROOT / 'val').exists() else SEG_ROOT / 'validation'
    explainer_image = next(iter(sorted([p for p in val_dir.iterdir() if p.suffix.lower() in {'.bmp', '.png', '.jpg', '.jpeg'}])), None)
    image_pil = Image.open(explainer_image).convert('RGB')
    transform = make_classification_transform(IMAGE_SIZE, augment=False)
    image_tensor = transform(image_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(image_tensor)
        class_weights = torch.sigmoid(logits)[0].detach().cpu().numpy()

    layercam = LayerCAM(model, device=device)
    try:
        fused_cam, per_class_cams, active_indices = generate_fused_cam(
            layercam, image_tensor, class_weights=class_weights, confidence_threshold=0.5
        )
    finally:
        layercam.close()

    image_denorm = tensor_to_pil(image_tensor[0].detach().cpu())
    image_rgb = np.array(image_denorm, dtype=np.uint8)
    debug_dir = MORPH_DEBUG_OUTPUT / explainer_image.stem
    bone_likelihood, bone_support = build_bone_guidance(
        image_rgb, fused_cam, seed_percentile=88, support_percentile=62,
        min_component_area=40, debug_dir=debug_dir,
    )
    prompt_map = fuse_cam_with_bone_guidance(fused_cam, bone_likelihood, bone_support)
    point_prompts = extract_point_prompts(prompt_map, cam_percentile=85, max_points=5, min_component_area=100, support_mask=bone_support)

    panels = [
        ('Original', image_rgb, None),
        ('LayerCAM', fused_cam, 'jet'),
        ('CAM + bone guidance', prompt_map, 'jet'),
        ('Enhanced grayscale', np.array(Image.open(debug_dir / 'bone_gray_enhanced.png')), 'gray'),
        ('Cortical edge response', np.array(Image.open(debug_dir / 'bone_edge_response.png')), 'gray'),
        ('Bone likelihood', np.array(Image.open(debug_dir / 'bone_likelihood.png')), 'gray'),
        ('Seed pixels', np.array(Image.open(debug_dir / 'bone_seeds.png')), 'gray'),
        ('Morphology support', np.array(Image.open(debug_dir / 'bone_support.png')), 'gray'),
    ]
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.ravel()
    for ax, (title, arr, cmap) in zip(axes, panels):
        ax.imshow(arr, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
    for r, c in point_prompts:
        axes[2].scatter([c], [r], c='red', s=70, edgecolors='white')
    plt.tight_layout()
    plt.show()
else:
    print('RUN_MORPHOLOGY_EXPLAINER=False or classifier checkpoint is missing.')

## 8. Quantitative Evaluation: Pseudo Mask vs RAM-H1200 Ground Truth

**Purpose:** measure pseudo-mask quality with Dice and IoU.

If `RUN_FULL_PSEUDO_MASKS=False`, this is only a 10-image preview metric.

In [ ]:
eval_cmd = [
    sys.executable, 'evaluate_ramh1200_masks.py',
    '--ram-root', str(RAM_ROOT),
    '--split', 'val',
    '--pred-mask-root', str(PSEUDO_OUTPUT / 'masks'),
    '--image-size', str(IMAGE_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--output-csv', str(EVAL_CSV),
]
print(' '.join(eval_cmd))
if RUN_EVALUATE_PSEUDO:
    subprocess.run(eval_cmd, check=True)
else:
    print('RUN_EVALUATE_PSEUDO=False; evaluation is skipped.')

if EVAL_CSV.exists():
    eval_df = pd.read_csv(EVAL_CSV)
    display(eval_df.tail(10))
    valid = eval_df[eval_df['status'] == 'ok'].copy()
    if not valid.empty:
        print('Images evaluated:', len(valid))
        print('Mean Dice:', valid['dice'].mean())
        print('Mean IoU :', valid['iou'].mean())
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        valid['dice'].hist(ax=axes[0], bins=20)
        axes[0].set_title('Pseudo-mask Dice distribution')
        valid['iou'].hist(ax=axes[1], bins=20)
        axes[1].set_title('Pseudo-mask IoU distribution')
        plt.tight_layout()
        plt.show()

## 9. Stage 3: Supervised U-Net Baseline

**Pipeline role:** train a fully supervised U-Net using RAM-H1200 ground-truth masks.

This gives a supervised reference point for the pseudo-mask branch.

In [ ]:
unet_cmd = [
    sys.executable, 'train_segmentation.py',
    '--ram-root', str(RAM_ROOT),
    '--train-split', 'train',
    '--val-split', 'val',
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE_SEGMENTATION),
    '--num-workers', str(NUM_WORKERS),
    '--epochs', str(EPOCHS_SEGMENTATION),
    '--output-dir', str(SEG_OUTPUT),
]
print(' '.join(unet_cmd))
if RUN_TRAIN_UNET:
    subprocess.run(unet_cmd, check=True)
else:
    print('RUN_TRAIN_UNET=False; U-Net training is skipped.')

seg_log = SEG_OUTPUT / 'training_log.csv'
seg_ckpt = SEG_OUTPUT / 'best_unet.pt'
print('U-Net checkpoint:', seg_ckpt, 'exists=', seg_ckpt.exists())
if seg_log.exists():
    seg_df = pd.read_csv(seg_log)
    display(seg_df.tail())
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    seg_df[['train_loss', 'val_loss']].plot(ax=axes[0], title='U-Net loss')
    seg_df[['train_dice', 'val_dice']].plot(ax=axes[1], title='U-Net Dice')
    seg_df[['train_iou', 'val_iou']].plot(ax=axes[2], title='U-Net IoU')
    for ax in axes:
        ax.set_xlabel('epoch')
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 10. Full Pipeline Visualization on One Image

This stage creates a pipeline strip with `visualize_pipeline.py`: original image, LayerCAM, bone guidance, prompts, SAM candidates, pseudo mask, and optionally U-Net output.

In [ ]:
val_dir = SEG_ROOT / 'val' if (SEG_ROOT / 'val').exists() else SEG_ROOT / 'validation'
sample_image = next(iter(sorted([p for p in val_dir.iterdir() if p.suffix.lower() in {'.bmp', '.png', '.jpg', '.jpeg'}])), None)
print('Sample image:', sample_image)

viz_cmd = [
    sys.executable, 'visualize_pipeline.py',
    '--image-path', str(sample_image),
    '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
    '--sam-checkpoint', str(SAM_CHECKPOINT),
    '--image-size', str(IMAGE_SIZE),
    '--selection-method', 'bone_hybrid',
    '--morphology-fusion-mode', 'components',
    '--sam-prompt-mode', 'box_point',
    '--output-path', str(VIZ_OUTPUT / f'{sample_image.stem}_pipeline.png'),
]
if (SEG_OUTPUT / 'best_unet.pt').exists():
    viz_cmd.extend(['--segmentation-checkpoint', str(SEG_OUTPUT / 'best_unet.pt')])
print(' '.join(viz_cmd))
if RUN_VISUALIZE_SAMPLE and sample_image is not None:
    subprocess.run(viz_cmd, check=True)

viz_files = sorted(VIZ_OUTPUT.glob('*_pipeline.png'))
if viz_files:
    img = Image.open(viz_files[-1])
    plt.figure(figsize=(22, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(viz_files[-1].name)
    plt.show()

## 11. SAM Candidate Debugging

This is the failure-analysis stage. It saves and displays every SAM candidate mask, each overlay, and `scores.json`.

In [ ]:
debug_image = sample_image
DEBUG_OUTPUT.mkdir(parents=True, exist_ok=True)
debug_strip = DEBUG_OUTPUT / f'{debug_image.stem}_debug_pipeline.png'

debug_cmd = [
    sys.executable, 'visualize_pipeline.py',
    '--image-path', str(debug_image),
    '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
    '--sam-checkpoint', str(SAM_CHECKPOINT),
    '--image-size', str(IMAGE_SIZE),
    '--selection-method', 'bone_hybrid',
    '--morphology-fusion-mode', 'components',
    '--sam-prompt-mode', 'box_point',
    '--debug',
    '--output-path', str(debug_strip),
]
print(' '.join(debug_cmd))
if RUN_DEBUG_SAM:
    subprocess.run(debug_cmd, check=True)

if debug_strip.exists():
    plt.figure(figsize=(22, 4))
    plt.imshow(Image.open(debug_strip))
    plt.axis('off')
    plt.title('Debug pipeline strip')
    plt.show()

import json
debug_dir = DEBUG_OUTPUT / 'debug' / debug_image.stem
mask_files = sorted(debug_dir.glob('mask_*.png'))
overlay_files = sorted(debug_dir.glob('overlay_mask_*.png'))
scores_path = debug_dir / 'scores.json'
print('debug_dir:', debug_dir)
print('SAM candidate masks:', len(mask_files))

if mask_files:
    n = len(mask_files)
    fig, axes = plt.subplots(2, n, figsize=(4*n, 7))
    if n == 1:
        axes = np.array([[axes[0]], [axes[1]]])
    for i, mf in enumerate(mask_files):
        axes[0][i].imshow(Image.open(mf), cmap='gray')
        axes[0][i].set_title(mf.stem)
        axes[0][i].axis('off')
        if i < len(overlay_files):
            axes[1][i].imshow(Image.open(overlay_files[i]).convert('RGB'))
        axes[1][i].set_title('overlay')
        axes[1][i].axis('off')
    plt.tight_layout()
    plt.show()

if scores_path.exists():
    scores = json.loads(scores_path.read_text(encoding='utf-8'))
    print('scores.json')
    for key, value in scores.items():
        score = value.get('score', 0)
        area = value.get('area', 0)
        bar = '#' * int(score * 30)
        print(f'{key:<8} score={score:.3f} area={area:>8} {bar}')

## 12. Small Ablation Preview

Optional. Enable `RUN_ABLATION_PREVIEW=True` to compare a few prompt/scoring strategies on the same image.

In [ ]:
ABLATION_OUTPUT.mkdir(parents=True, exist_ok=True)
ABLATION_STRATEGIES = [
    ('box_point_bone_hybrid', ['--sam-prompt-mode', 'box_point', '--selection-method', 'bone_hybrid']),
    ('point_bone_hybrid', ['--sam-prompt-mode', 'point', '--selection-method', 'bone_hybrid']),
    ('box_point_coverage', ['--sam-prompt-mode', 'box_point', '--selection-method', 'coverage']),
]

if RUN_ABLATION_PREVIEW:
    for label, extra in ABLATION_STRATEGIES:
        out_path = ABLATION_OUTPUT / f'{label}_pipeline.png'
        cmd = [
            sys.executable, 'visualize_pipeline.py',
            '--image-path', str(debug_image),
            '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
            '--sam-checkpoint', str(SAM_CHECKPOINT),
            '--image-size', str(IMAGE_SIZE),
            '--morphology-fusion-mode', 'components',
            '--output-path', str(out_path),
            *extra,
        ]
        print(' '.join(cmd))
        subprocess.run(cmd, check=True)

ablation_files = sorted(ABLATION_OUTPUT.glob('*_pipeline.png'))
if ablation_files:
    fig, axes = plt.subplots(len(ablation_files), 1, figsize=(22, 4*len(ablation_files)))
    if len(ablation_files) == 1:
        axes = [axes]
    for ax, path in zip(axes, ablation_files):
        ax.imshow(Image.open(path))
        ax.set_title(path.stem)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 13. Reporting Checklist

Save these artifacts for the thesis/report:

1. Dataset sample: original X-ray + GT bone mask + overlay.
2. Stage 2 sample: original + LayerCAM overlay + pseudo mask.
3. Morphology explainer: grayscale, edge, likelihood, seeds, support, prompt points.
4. Pipeline strip from `visualize_pipeline.py`.
5. Dice/IoU table from `evaluate_ramh1200_masks.py`.
6. Supervised U-Net training curves and best validation Dice/IoU.
7. Optional ablation figures for prompt mode or scoring method.